In [1]:
from solver import *
from theta_vals import theta_sols_C1
import matplotlib.pyplot as plt
import scipy.optimize as opt

In [2]:
theta_amp_initial = theta_sols_C1[90]
def make_theta_phase_init(
    theta_amp_init,
    p_phase=None,
    perturbation=0.0,
    seed=0,
):
    """
    Initialize the direct phase Ansatz

        omega(V) = exp(log_kappa) * V + x(V)
    
    at, or very close to, fixed phase omega(V)=V.

    Parameter ordering:
        [log_kappa, a0, a, w, b]
    """
    theta_amp_init = np.asarray(
        theta_amp_init,
        dtype=np.float64,
    )

    if (theta_amp_init.size - 1) % 3 != 0:
        raise ValueError(
            "theta_amp_init must have length 1 + 3*p."
        )

    p_amp = (theta_amp_init.size - 1) // 3
    p = p_amp if p_phase is None else int(p_phase)

    if p < 1:
        raise ValueError("p_phase must be positive.")

    # Tanh transition centres distributed across the interval.
    centres = np.linspace(0.15, 0.85, p)

    # Moderate slopes: neither almost linear nor excessively sharp.
    widths = np.linspace(4.0, 12.0, p)

    # w_j V + b_j = 0 at V = centre_j.
    biases = -widths * centres

    rng = np.random.default_rng(seed)

    # Zero gives exactly omega=V.
    # A tiny value such as 1e-4 gives a small symmetry-breaking seed.
    a0 = perturbation * rng.normal()
    a = perturbation * rng.normal(size=p)

    theta_phase_init = np.concatenate([
        np.array([0.0]),  # log_kappa=0, hence kappa=1
        np.array([a0]),
        a,
        widths,
        biases,
    ])

    assert theta_phase_init.size == 3 * p + 2

    return theta_phase_init

theta_phase_initial = make_theta_phase_init(theta_amp_initial,perturbation=0.0)
eM_trial = 85

In [3]:
dtype = torch.float64
device = torch.device("cpu")

theta_1 = torch.nn.Parameter(torch.as_tensor(theta_amp_initial, dtype=dtype,device=device).clone())
theta_2 = torch.nn.Parameter(torch.as_tensor(theta_phase_initial, dtype=dtype,device=device).clone())

In [4]:
def objective_C1():
    return loss_C1(theta_1, theta_2, eM_trial, q=1, rho_drho_func=tanh_ansatz, omega_domega_func=omega_domega)

In [5]:
result = adam_optimize(
    parameters=[theta_1, theta_2],
    loss_fn=objective_C1,
    lr=1e-3,
    max_iter=1000,
    loss_tol=1e-12,
    grad_tol=1e-8,
    patience=200,
    min_delta=1e-12,
    max_grad_norm=None,
    print_every=20,
)

step=    0, loss=3.086542e-03, best=3.086542e-03, |g|=1.950989e-01
step=   20, loss=3.543188e-05, best=3.543188e-05, |g|=1.998730e-02
step=   40, loss=4.985844e-05, best=3.197518e-05, |g|=1.333056e-02
step=   60, loss=2.559264e-05, best=2.346434e-05, |g|=6.680451e-03
step=   80, loss=1.684717e-05, best=1.684717e-05, |g|=2.442992e-03
step=  100, loss=1.248480e-05, best=1.248480e-05, |g|=1.505695e-03
step=  120, loss=9.141084e-06, best=9.141084e-06, |g|=1.122303e-03
step=  140, loss=6.492778e-06, best=6.492778e-06, |g|=9.654220e-04
step=  160, loss=4.468767e-06, best=4.468767e-06, |g|=7.834420e-04
step=  180, loss=2.982060e-06, best=2.982060e-06, |g|=6.480657e-04
step=  200, loss=1.930578e-06, best=1.930578e-06, |g|=5.202708e-04
step=  220, loss=1.213197e-06, best=1.213197e-06, |g|=4.138055e-04
step=  240, loss=7.403600e-07, best=7.403600e-07, |g|=3.234641e-04
step=  260, loss=4.389194e-07, best=4.389194e-07, |g|=2.492901e-04
step=  280, loss=2.528649e-07, best=2.528649e-07, |g|=1.893897

In [ ]:
best_theta_1_torch, best_theta_2_torch = result["best_params"]

best_theta_1 = (best_theta_1_torch.detach().cpu().numpy())
best_theta_2 = (best_theta_2_torch.detach().cpu().numpy())

print("best loss:", result["best_loss"])
print("stop reason:", result["stop_reason"])

best loss: 1.6119207959887824e-12
stop reason: loss tolerance reached
